In [2]:
# 环境检查
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch: 2.9.0+cpu
CUDA: False
Device: cpu


In [ ]:
# 检查依赖 + 安装 TorchInspector
try:
    import pyarrow
    print(f"pyarrow: {pyarrow.__version__}")
except ImportError:
    !pip install pyarrow -q

# 清除残留的空命名空间包
import sys
for k in list(sys.modules):
    if k.startswith('torchinspector'):
        del sys.modules[k]

!rm -rf /content/torchinspector
!git clone https://github.com/blackcat312340/torchinspector.git /content/torchinspector
!pip install /content/torchinspector -q
!rm -rf /content/torchinspector

from torchinspector import Inspector
print("torchinspector: OK")

In [ ]:
import time
from pathlib import Path
from copy import deepcopy
from itertools import product

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchinspector import Inspector

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. 下载数据

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

PARQUET_NAMES = [
    "yellow_tripdata_2024-07.parquet",
    "yellow_tripdata_2024-08.parquet",
    "yellow_tripdata_2024-09.parquet",
    "yellow_tripdata_2024-10.parquet",
    "yellow_tripdata_2024-11.parquet",
    "yellow_tripdata_2024-12.parquet",
    "yellow_tripdata_2025-01.parquet",
]
TLC_BASE = "https://d37ci6vzurychx.cloudfront.net/trip-data/"

import urllib.request
for name in PARQUET_NAMES:
    path = DATA_DIR / name
    if path.exists():
        print(f"  [OK] {name}")
        continue
    print(f"  [DL] {name} ...", end="", flush=True)
    urllib.request.urlretrieve(TLC_BASE + name, path)
    print(f" ({path.stat().st_size / 1024 / 1024:.1f} MB)")

## 2. 数据处理 + 特征工程

In [ ]:
def load_and_merge():
    dfs = []
    for name in PARQUET_NAMES:
        df = pd.read_parquet(DATA_DIR / name, columns=["tpep_pickup_datetime"])
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

def build_hourly_series(df):
    df = df.copy()
    df["pickup_dt"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
    df = df.dropna(subset=["pickup_dt"])
    df["hour_bin"] = df["pickup_dt"].dt.floor("h")
    demand = df.groupby("hour_bin").size().reset_index(name="demand")
    return demand.sort_values("hour_bin").reset_index(drop=True)

def add_features(demand):
    demand = demand.copy()
    demand["hour"] = demand["hour_bin"].dt.hour
    demand["weekday"] = demand["hour_bin"].dt.weekday
    demand["is_peak"] = ((demand["hour"].between(7, 10)) | (demand["hour"].between(16, 20))).astype(float)
    demand["is_weekend"] = (demand["weekday"] >= 5).astype(float)
    demand["hour_sin"] = np.sin(2 * np.pi * demand["hour"] / 24)
    demand["hour_cos"] = np.cos(2 * np.pi * demand["hour"] / 24)
    demand["weekday_sin"] = np.sin(2 * np.pi * demand["weekday"] / 7)
    demand["weekday_cos"] = np.cos(2 * np.pi * demand["weekday"] / 7)
    demand["lag_24h"] = demand["demand"].shift(24).bfill()
    demand["lag_168h"] = demand["demand"].shift(168).bfill()
    demand["roll_mean_6h"] = demand["demand"].rolling(6, min_periods=1).mean()
    demand["roll_std_6h"] = demand["demand"].rolling(6, min_periods=1).std().fillna(0)
    demand["roll_mean_24h"] = demand["demand"].rolling(24, min_periods=1).mean()
    demand["month"] = demand["hour_bin"].dt.month
    demand["month_sin"] = np.sin(2 * np.pi * demand["month"] / 12)
    demand["month_cos"] = np.cos(2 * np.pi * demand["month"] / 12)
    demand["demand_diff_1h"] = demand["demand"].diff(1).fillna(0)
    demand["demand_diff_24h"] = demand["demand"].diff(24).fillna(0)
    return demand

def normalize(series):
    m, s = series.mean(), series.std()
    return (series - m) / s, m, max(s, 1)

def build_sequences(demand, feature_cols, seq_len):
    features = demand[feature_cols].values
    targets = demand["demand_norm"].values
    X, y = [], []
    for i in range(len(demand) - seq_len):
        X.append(features[i:i + seq_len])
        y.append(targets[i + seq_len])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

# 执行
print("加载数据...")
df = load_and_merge()
demand = build_hourly_series(df)
demand = add_features(demand)
demand["demand_norm"], demand_mean, demand_std = normalize(demand["demand"])

num_cols = ["lag_24h", "lag_168h", "roll_mean_6h", "roll_std_6h", "roll_mean_24h", "demand_diff_1h", "demand_diff_24h"]
for c in num_cols:
    m, s = demand[c].mean(), demand[c].std()
    demand[c] = (demand[c] - m) / max(s, 1)

FEATURE_COLS = [
    "demand_norm", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
    "is_peak", "is_weekend", "lag_24h", "lag_168h",
    "roll_mean_6h", "roll_std_6h", "roll_mean_24h",
    "month_sin", "month_cos", "demand_diff_1h", "demand_diff_24h",
]
print(f"{len(demand)} 小时 | {len(FEATURE_COLS)} 特征 | 均值 {demand_mean:.0f}")

## 3. 模型 + 训练工具

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True)
        self.norm = nn.LayerNorm(hidden_dim * 2)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(self.norm(out[:, -1, :])).squeeze(-1)


class EMA:
    def __init__(self, model, decay=0.998):
        self.decay = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)

    def apply(self, model):
        self.backup = {k: v.clone() for k, v in model.state_dict().items()}
        model.load_state_dict(self.shadow)

    def restore(self, model):
        model.load_state_dict(self.backup)


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup, total):
        self.optimizer = optimizer
        self.warmup = warmup
        self.total = total
        self.base_lr = optimizer.param_groups[0]["lr"]

    def step(self, epoch):
        if epoch < self.warmup:
            lr = self.base_lr * (epoch + 1) / self.warmup
        else:
            p = (epoch - self.warmup) / max(1, self.total - self.warmup)
            lr = self.base_lr * 0.5 * (1 + np.cos(np.pi * p))
        for pg in self.optimizer.param_groups:
            pg["lr"] = lr


def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0))
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


class LogCoshLoss(nn.Module):
    def forward(self, pred, target):
        return torch.mean(torch.log(torch.cosh(pred - target + 1e-12)))

## 4. 超参搜索 + Loss 对比

In [ ]:
def train_once(X_train, y_train, X_test, y_test, demand_std, demand_mean,
               hidden_dim, seq_len, lr, loss_fn, epochs=80, log_tag=""):
    model = BiLSTM(X_train.shape[2], hidden_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = WarmupCosineScheduler(optimizer, warmup=5, total=epochs)
    ema = EMA(model)

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=256, shuffle=True)
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=256)

    log_dir = f"runs/{log_tag}"
    with Inspector(model, optimizer, log_dir,
                   log_interval=30, rnn_interval=30, health_report_interval=100) as ins:
        ins.watch_auto(max_layers=8)

        best_mae = float("inf")
        for epoch in range(1, epochs + 1):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                xb_mix, yb_a, yb_b, lam = mixup_data(xb, yb, 0.2)
                pred = model(xb_mix)
                loss = lam * loss_fn(pred, yb_a) + (1 - lam) * loss_fn(pred, yb_b)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                ema.update(model)
                ins.step(loss=loss.item())
            scheduler.step(epoch)

            ema.apply(model)
            model.eval()
            v_mae, v_cnt = 0, 0
            with torch.no_grad():
                for xb, yb in test_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    v_mae += (model(xb) - yb).abs().mean().item() * len(xb)
                    v_cnt += len(xb)
            ema.restore(model)

            mae_real = (v_mae / v_cnt) * demand_std
            best_mae = min(best_mae, mae_real)
            if epoch % 20 == 0 or epoch == 1:
                print(f"  Epoch {epoch:3d} | MAE: {mae_real:.0f} ({mae_real/demand_mean*100:.1f}%)")

    return best_mae

In [ ]:
# ── 实验1: Loss 对比 ──
print("=" * 50)
print("Loss 对比 (hidden=256, seq=48, lr=8e-4)")
print("=" * 50)

X, y = build_sequences(demand, FEATURE_COLS, 48)
n = len(X)
idx = torch.randperm(n)
split = int(0.8 * n)
X_train, y_train = X[idx[:split]].to(device), y[idx[:split]].to(device)
X_test, y_test = X[idx[split:]].to(device), y[idx[split:]].to(device)

loss_fns = {
    "huber": nn.HuberLoss(delta=0.5),
    "logcosh": LogCoshLoss(),
    "mse": nn.MSELoss(),
    "mae": nn.L1Loss(),
}

loss_results = {}
for name, fn in loss_fns.items():
    print(f"\n--- {name} ---")
    mae = train_once(X_train, y_train, X_test, y_test,
                     demand_std, demand_mean, 256, 48, 8e-4, fn,
                     epochs=80, log_tag=f"loss_{name}")
    loss_results[name] = mae
    print(f">>> {name}: MAE = {mae:.0f} ({mae/demand_mean*100:.1f}%)")

print("\n" + "=" * 50)
print("Loss 对比结果")
print("=" * 50)
for name, mae in sorted(loss_results.items(), key=lambda x: x[1]):
    print(f"  {name:<10} MAE: {mae:.0f} ({mae/demand_mean*100:.1f}%)")

In [ ]:
# ── 实验2: 超参搜索 ──
best_loss_name = min(loss_results, key=loss_results.get)
best_loss_fn = loss_fns[best_loss_name]
print(f"最优 Loss: {best_loss_name}")

search_space = [
    (128, 48, 5e-4),
    (128, 48, 8e-4),
    (256, 48, 5e-4),
    (256, 48, 8e-4),
    (384, 48, 6e-4),
    (256, 24, 8e-4),
    (256, 72, 8e-4),
]

search_results = []
for hdim, slen, lr in search_space:
    tag = f"h{hdim}_s{slen}_lr{lr}"
    print(f"\n--- {tag} ---")

    X, y = build_sequences(demand, FEATURE_COLS, slen)
    n = len(X)
    idx = torch.randperm(n)
    split = int(0.8 * n)
    X_tr, y_tr = X[idx[:split]].to(device), y[idx[:split]].to(device)
    X_te, y_te = X[idx[split:]].to(device), y[idx[split:]].to(device)

    mae = train_once(X_tr, y_tr, X_te, y_te,
                     demand_std, demand_mean, hdim, slen, lr, best_loss_fn,
                     epochs=80, log_tag=tag)
    search_results.append((hdim, slen, lr, mae))
    print(f">>> {tag}: MAE = {mae:.0f} ({mae/demand_mean*100:.1f}%)")

# 汇总
search_results.sort(key=lambda x: x[3])
print("\n" + "=" * 50)
print("超参搜索结果排名")
print("=" * 50)
print(f"  {'hidden':>6} {'seq':>4} {'lr':>8} {'MAE':>8} {'误差':>6}")
for hdim, slen, lr, mae in search_results:
    print(f"  {hdim:>6} {slen:>4} {lr:>8.1e} {mae:>7.0f} {mae/demand_mean*100:>5.1f}%")

best = search_results[0]
print(f"\n最优配置: hidden={best[0]}, seq={best[1]}, lr={best[2]}, MAE={best[3]:.0f} ({best[3]/demand_mean*100:.1f}%)")

## 5. TensorBoard

In [ ]:
# 6. 汇总输出
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

for run in sorted(glob.glob("runs/*")):
    ea = EventAccumulator(run)
    ea.Reload()
    tags = ea.Tags()
    scalars = sorted(tags.get("scalars", []))
    images = sorted(tags.get("images", []))
    hists = sorted(tags.get("histograms", []))

    print(f"\n{'='*60}")
    print(f"  {run}")
    print(f"{'='*60}")

    if scalars:
        print(f"  Scalars ({len(scalars)}):")
        for t in scalars[:8]:
            e = ea.Scalars(t)
            if e:
                v = [x.value for x in e]
                print(f"    {t}: {v[0]:.4f} -> {v[-1]:.4f} ({len(v)} pts)")
        if len(scalars) > 8:
            print(f"    ... +{len(scalars)-8} more")

    if images:
        print(f"  Images ({len(images)}):")
        for t in images:
            print(f"    {t}: {len(ea.Images(t))} imgs")

    if hists:
        print(f"  Histograms ({len(hists)}):")
        for t in hists[:5]:
            print(f"    {t}")
        if len(hists) > 5:
            print(f"    ... +{len(hists)-5} more")

In [ ]:
# 6. 汇总输出
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

for run in sorted(glob.glob("runs/*")):
    ea = EventAccumulator(run)
    ea.Reload()
    tags = ea.Tags()
    scalars = sorted(tags.get("scalars", []))
    images = sorted(tags.get("images", []))
    hists = sorted(tags.get("histograms", []))
    print(f"\n{'='*60}")
    print(f"  {run}")
    print(f"{'='*60}")
    if scalars:
        print(f"  Scalars ({len(scalars)}):")
        for t in scalars[:8]:
            e = ea.Scalars(t)
            if e:
                v = [x.value for x in e]
                print(f"    {t}: {v[0]:.4f} -> {v[-1]:.4f} ({len(v)} pts)")
        if len(scalars) > 8:
            print(f"    ... +{len(scalars)-8} more")
    if images:
        print(f"  Images ({len(images)}):")
        for t in images:
            print(f"    {t}: {len(ea.Images(t))} imgs")
    if hists:
        print(f"  Histograms ({len(hists)}):")
        for t in hists[:5]:
            print(f"    {t}")
        if len(hists) > 5:
            print(f"    ... +{len(hists)-5} more")

In [ ]:
# 6. 汇总输出
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

for run in sorted(glob.glob("runs/*")):
    ea = EventAccumulator(run)
    ea.Reload()
    tags = ea.Tags()
    scalars = sorted(tags.get("scalars", []))
    images = sorted(tags.get("images", []))
    hists = sorted(tags.get("histograms", []))
    print(f"\n{'='*60}")
    print(f"  {run}")
    print(f"{'='*60}")
    if scalars:
        print(f"  Scalars ({len(scalars)}):")
        for t in scalars[:8]:
            e = ea.Scalars(t)
            if e:
                v = [x.value for x in e]
                print(f"    {t}: {v[0]:.4f} -> {v[-1]:.4f} ({len(v)} pts)")
        if len(scalars) > 8:
            print(f"    ... +{len(scalars)-8} more")
    if images:
        print(f"  Images ({len(images)}):")
        for t in images:
            print(f"    {t}: {len(ea.Images(t))} imgs")
    if hists:
        print(f"  Histograms ({len(hists)}):")
        for t in hists[:5]:
            print(f"    {t}")
        if len(hists) > 5:
            print(f"    ... +{len(hists)-5} more")

In [ ]:
# 6. 汇总输出
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import glob

for run in sorted(glob.glob("runs/*")):
    ea = EventAccumulator(run)
    ea.Reload()
    tags = ea.Tags()
    scalars = sorted(tags.get("scalars", []))
    images = sorted(tags.get("images", []))
    hists = sorted(tags.get("histograms", []))
    print(f"\n{'='*60}")
    print(f"  {run}")
    print(f"{'='*60}")
    if scalars:
        print(f"  Scalars ({len(scalars)}):")
        for t in scalars[:8]:
            e = ea.Scalars(t)
            if e:
                v = [x.value for x in e]
                print(f"    {t}: {v[0]:.4f} -> {v[-1]:.4f} ({len(v)} pts)")
        if len(scalars) > 8:
            print(f"    ... +{len(scalars)-8} more")
    if images:
        print(f"  Images ({len(images)}):")
        for t in images:
            print(f"    {t}: {len(ea.Images(t))} imgs")
    if hists:
        print(f"  Histograms ({len(hists)}):")
        for t in hists[:5]:
            print(f"    {t}")
        if len(hists) > 5:
            print(f"    ... +{len(hists)-5} more")